# SVG Inference — Load Merged Model & Generate Submission

Inference-only notebook. Loads the pre-trained merged model from Kaggle kernel output and generates `submission.csv` for the 1,000 test prompts.

**Before running:** Add the training kernel output as a data source in Kaggle (Add Data → Kernel Outputs → search `ryanmfleishman/dl-llama-version`). The merged model will then be at `/kaggle/input/dl-llama-version/qwen25_coder_svg_lora_merged`.

In [1]:
%pip install -q unsloth transformers accelerate peft bitsandbytes pandas lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths
MERGED_MODEL_DIR = "/kaggle/input/datasets/ryanmfleishman/dl-training/qwen25_coder_svg_lora_merged"
TRAIN_PATH       = "/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv"
TEST_PATH        = "/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"
MAX_NEW_TOKENS   = 1024

RETRIEVAL_THRESHOLD = 0.6  # cosine similarity above this → use training SVG directly

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Model path exists: {os.path.isdir(MERGED_MODEL_DIR)}")

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Model path exists: True


In [3]:
from lxml import etree

# SVG constraint constants
MAX_SVG_CHARS     = 16_000
MAX_PATH_ELEMENTS = 256
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "defs", "use", "symbol", "clipPath",
    "mask", "linearGradient", "radialGradient", "stop", "text",
    "tspan", "title", "desc", "style", "pattern", "marker", "filter",
}


def _local_tag(element):
    tag = element.tag
    return tag.split("}", 1)[-1] if isinstance(tag, str) and "}" in tag else tag


def is_valid_svg(svg_text):
    if not svg_text or not isinstance(svg_text, str):
        return False
    if len(svg_text) > MAX_SVG_CHARS:
        return False
    svg_text = svg_text.strip()
    if not svg_text.lower().startswith("<svg"):
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if _local_tag(root) != "svg":
        return False
    path_count = 0
    for el in root.iter():
        tag = _local_tag(el)
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1
    if path_count > MAX_PATH_ELEMENTS:
        return False
    return True


def normalize_dimensions(svg_text):
    """
    Force width, height, and viewBox to 256x256 on the root <svg> element.

    The model sometimes outputs other canvas sizes (128, 400, etc.). The
    competition renders everything on a 256x256 canvas, so wrong dimensions
    directly hurt SSIM scores.

    If the original viewBox has a non-square or non-256 coordinate system we
    keep the internal coordinate space and let the width/height scaling handle
    the rest — SVG viewers scale content to fit width/height automatically.
    """
    if not svg_text:
        return svg_text
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg_text.encode("utf-8"), parser)
    except Exception:
        return svg_text
    if _local_tag(root) != "svg":
        return svg_text
    root.set("width", "256")
    root.set("height", "256")
    # Only override viewBox if it's missing or clearly wrong (not 0 0 256 256)
    vb = root.get("viewBox", "")
    if not vb:
        root.set("viewBox", "0 0 256 256")
    return etree.tostring(root, encoding="unicode")


# ---------------------------------------------------------------------------
# SVG repair: salvage malformed model output before falling back.
# ---------------------------------------------------------------------------
def repair_svg(raw_text):
    if not raw_text or not isinstance(raw_text, str):
        return None

    text = raw_text.strip()

    # 1. Locate the opening <svg tag
    m = re.search(r"<svg[\s>]", text, re.IGNORECASE)
    if not m:
        return None
    text = text[m.start():]

    # 2. If closing tag is missing the model was truncated — append it
    if not re.search(r"</svg\s*>", text, re.IGNORECASE):
        text = text + "</svg>"

    # 3. lxml recovery parse handles most remaining XML errors
    try:
        parser = etree.XMLParser(recover=True, remove_comments=True)
        root = etree.fromstring(text.encode("utf-8"), parser)
    except Exception:
        return None
    if root is None or _local_tag(root) != "svg":
        return None

    # 4. Ensure required root attributes
    if "xmlns" not in root.attrib:
        root.set("xmlns", "http://www.w3.org/2000/svg")
    for attr, val in [("width", "256"), ("height", "256"), ("viewBox", "0 0 256 256")]:
        if attr not in root.attrib:
            root.set(attr, val)

    # 5. Remove disallowed tags
    for el in reversed(root.xpath(".//*")):
        if _local_tag(el) not in ALLOWED_TAGS:
            parent = el.getparent()
            if parent is not None:
                parent.remove(el)

    # 6. Trim excess <path> elements
    paths = root.xpath(".//*[local-name()='path']")
    if len(paths) > MAX_PATH_ELEMENTS:
        for el in paths[MAX_PATH_ELEMENTS:]:
            p = el.getparent()
            if p is not None:
                p.remove(el)

    repaired = etree.tostring(root, encoding="unicode")
    return repaired if is_valid_svg(repaired) else None


# ---------------------------------------------------------------------------
# Prompt-aware fallback: only reached if generation AND repair both fail.
# ---------------------------------------------------------------------------
_COLOR_MAP = {
    "red": "red", "blue": "blue", "green": "green", "yellow": "yellow",
    "orange": "orange", "purple": "purple", "pink": "pink",
    "black": "black", "white": "white", "gray": "gray", "grey": "gray",
    "brown": "#8B4513", "cyan": "cyan", "magenta": "magenta",
    "gold": "gold", "silver": "silver", "teal": "teal", "navy": "navy",
}

def prompt_aware_fallback(prompt=""):
    p = prompt.lower()

    fill = "steelblue"
    for kw, color in _COLOR_MAP.items():
        if kw in p:
            fill = color
            break

    bg = "black" if any(w in p for w in ("dark", "night", "space", "black background")) else "white"

    if any(w in p for w in ("circle", "ball", "sphere", "sun", "moon", "dot")):
        shape = f'<circle cx="128" cy="128" r="80" fill="{fill}"/>'
    elif any(w in p for w in ("triangle", "pyramid", "mountain")):
        shape = f'<polygon points="128,32 224,214 32,214" fill="{fill}"/>'
    elif "star" in p:
        shape = f'<polygon points="128,28 148,88 212,88 160,124 180,188 128,150 76,188 96,124 44,88 108,88" fill="{fill}"/>'
    elif "heart" in p:
        shape = f'<path d="M128,195 C55,148 18,98 58,58 C78,38 108,43 128,68 C148,43 178,38 198,58 C238,98 201,148 128,195 Z" fill="{fill}"/>'
    elif any(w in p for w in ("square", "box", "rect", "flag", "window")):
        shape = f'<rect x="48" y="48" width="160" height="160" fill="{fill}"/>'
    elif any(w in p for w in ("ellipse", "oval")):
        shape = f'<ellipse cx="128" cy="128" rx="110" ry="65" fill="{fill}"/>'
    elif any(w in p for w in ("line", "diagonal", "stripe")):
        shape = f'<line x1="24" y1="128" x2="232" y2="128" stroke="{fill}" stroke-width="10"/>'
    elif any(w in p for w in ("text", "letter", "word", "number")):
        shape = f'<text x="128" y="148" font-size="64" text-anchor="middle" fill="{fill}">A</text>'
    else:
        shape = f'<circle cx="128" cy="128" r="80" fill="{fill}"/>'

    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        f'<rect width="256" height="256" fill="{bg}"/>'
        f'{shape}'
        f'</svg>'
    )


# Sanity checks
assert is_valid_svg(prompt_aware_fallback("a red circle")), "Fallback must be valid"
assert is_valid_svg(prompt_aware_fallback()), "Default fallback must be valid"
_fixed = normalize_dimensions('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 400 400" height="128" width="128"><circle cx="200" cy="200" r="180" fill="blue"/></svg>')
assert 'width="256"' in _fixed and 'height="256"' in _fixed, "normalize_dimensions must set 256x256"
print("SVG helpers ready (repair + normalize_dimensions + prompt-aware fallback).")

SVG helpers ready (repair + normalize_dimensions + prompt-aware fallback).


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="cuda:0",
)
model.eval()

# torch.compile gives ~20-40% throughput improvement on T4 with a one-time
# ~30s compilation cost on the first generate call.
model = torch.compile(model, mode="reduce-overhead")

print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

_tok = tokenizer("warmup", return_tensors="pt").to("cuda:0")
with torch.no_grad():
    _t0 = time.time()
    model.generate(**_tok, max_new_tokens=32, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    _elapsed = time.time() - _t0
print(f"Warmup (includes first-run compile): {_elapsed:.2f}s for 32 tokens  ({32/_elapsed:.1f} tok/s)")

# Second run shows true compiled speed
with torch.no_grad():
    _t0 = time.time()
    model.generate(**_tok, max_new_tokens=32, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    _elapsed = time.time() - _t0
print(f"Warmup (compiled, 2nd run):           {_elapsed:.2f}s for 32 tokens  ({32/_elapsed:.1f} tok/s)")

The tokenizer you are loading from '/kaggle/input/datasets/ryanmfleishman/dl-training/qwen25_coder_svg_lora_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

GPU memory used: 3.09 GB / 15.64 GB
Warmup (includes first-run compile): 2.78s for 32 tokens  (11.5 tok/s)
Warmup (compiled, 2nd run):           1.26s for 32 tokens  (25.3 tok/s)


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# --- Retrieval index ---
print("Loading training data for retrieval...")
train_df = pd.read_csv(TRAIN_PATH).dropna(subset=["prompt", "svg"])
train_df["prompt"] = train_df["prompt"].str.strip()
train_df["svg"]    = train_df["svg"].str.strip()
valid_mask = train_df["svg"].apply(is_valid_svg)
train_df = train_df[valid_mask].reset_index(drop=True)
print(f"Valid training examples: {len(train_df)}")

vectorizer    = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
train_vectors = vectorizer.fit_transform(train_df["prompt"])
print(f"TF-IDF index ready. Vocab size: {len(vectorizer.vocabulary_)}")


def retrieve_svg(prompt):
    vec      = vectorizer.transform([prompt])
    sims     = cos_sim(vec, train_vectors)[0]
    best_idx = int(sims.argmax())
    best_sim = float(sims[best_idx])
    if best_sim >= RETRIEVAL_THRESHOLD:
        return normalize_dimensions(train_df.iloc[best_idx]["svg"]), best_sim
    return None, best_sim


# --- Model generation ---
SYSTEM_PROMPT = (
    "You generate valid SVG markup from a natural language description. "
    "Output only the SVG code, nothing else.\n\n"
    "Requirements:\n"
    "- Root element must be exactly: <svg xmlns=\"http://www.w3.org/2000/svg\" width=\"256\" height=\"256\" viewBox=\"0 0 256 256\">\n"
    "- Allowed tags only: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, "
    "title, desc, style, pattern, marker, filter\n"
    "- Maximum 16000 characters total\n"
    "- Maximum 256 <path> elements\n\n"
    "Example:\n"
    "User: a blue circle on a white background\n"
    "Assistant: <svg xmlns=\"http://www.w3.org/2000/svg\" width=\"256\" height=\"256\" "
    "viewBox=\"0 0 256 256\"><rect width=\"256\" height=\"256\" fill=\"white\"/>"
    "<circle cx=\"128\" cy=\"128\" r=\"100\" fill=\"blue\"/></svg>"
)

SVG_REGEX = re.compile(r"<svg.*?</svg>", flags=re.IGNORECASE | re.DOTALL)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def model_generate_svg(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    input_len = inputs["input_ids"].shape[1]
    decoded   = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)

    svg = extract_svg(decoded)
    if is_valid_svg(svg):
        return normalize_dimensions(svg)
    repaired = repair_svg(decoded)
    if repaired:
        return normalize_dimensions(repaired)
    return prompt_aware_fallback(prompt)


def generate_svg(prompt):
    """Hybrid: retrieval when confident, model otherwise."""
    retrieved, sim = retrieve_svg(prompt)
    if retrieved is not None:
        return retrieved, "retrieval", sim
    return model_generate_svg(prompt), "model", sim


# Smoke test
_svg, _source, _sim = generate_svg("a simple blue circle on a white background")
print(f"\nSmoke test — source: {_source} (sim={_sim:.3f}) | valid: {is_valid_svg(_svg)} | length: {len(_svg)}")
print(_svg[:300])

Loading training data for retrieval...
Valid training examples: 49999
TF-IDF index ready. Vocab size: 80447

Smoke test — source: retrieval (sim=0.583) | valid: True | length: 411
<svg xmlns="http://www.w3.org/2000/svg" height="256" viewBox="0 0 1000 1000" width="256"><path d="m499 83q-113 0-210 57-94 55-149 149-57 97-57 210t57 210q55 94 149 149 97 57 210 57t210-57q94-55 149-149 57-97 57-210t-57-210q-55-94-149-149-97-57-210-57zm0 749q-90 0-168-46-75-44-119-119-46-78-46-168t46


In [6]:
test_df = pd.read_csv(TEST_PATH)
print(f"Test rows: {len(test_df)}")

all_ids     = test_df["id"].tolist()
all_prompts = test_df["prompt"].tolist()

rows = []
invalid_count    = 0
retrieval_count  = 0
t0 = time.time()

for i, (id_, prompt) in enumerate(zip(all_ids, all_prompts)):
    t_row = time.time()
    svg, source, sim = generate_svg(prompt)
    row_sec = time.time() - t_row

    if not is_valid_svg(svg):
        invalid_count += 1
    if source == "retrieval":
        retrieval_count += 1
    rows.append({"id": id_, "svg": svg})

    elapsed = (time.time() - t0) / 60
    rate    = (i + 1) / elapsed if elapsed > 0 else 0
    eta     = (len(all_prompts) - i - 1) / rate if rate > 0 else 0
    print(f"  [{i+1:4d}/{len(all_prompts)}] {row_sec:.1f}s | {source:9s} sim={sim:.2f} | "
          f"{elapsed:.1f} min elapsed | ETA {eta:.1f} min | "
          f"retrieved: {retrieval_count} | fallbacks: {invalid_count}")

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"\nSaved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Retrieved from training data: {retrieval_count} ({100*retrieval_count/len(sub_df):.1f}%)")
print(f"Generated by model:           {len(sub_df)-retrieval_count} ({100*(len(sub_df)-retrieval_count)/len(sub_df):.1f}%)")
print(f"Invalid/fallbacks:            {invalid_count}")
print(f"Runtime: {elapsed_min:.1f} min")
sub_df.head()

Test rows: 1000
  [   1/1000] 0.0s | retrieval sim=0.49 | 0.0 min elapsed | ETA 0.6 min | retrieved: 1 | fallbacks: 0
  [   2/1000] 0.0s | retrieval sim=0.72 | 0.0 min elapsed | ETA 0.6 min | retrieved: 2 | fallbacks: 0
  [   3/1000] 0.0s | retrieval sim=0.73 | 0.0 min elapsed | ETA 0.5 min | retrieved: 3 | fallbacks: 0
  [   4/1000] 0.0s | retrieval sim=0.44 | 0.0 min elapsed | ETA 0.5 min | retrieved: 4 | fallbacks: 0
  [   5/1000] 0.0s | retrieval sim=0.55 | 0.0 min elapsed | ETA 0.5 min | retrieved: 5 | fallbacks: 0
  [   6/1000] 0.0s | retrieval sim=0.45 | 0.0 min elapsed | ETA 0.5 min | retrieved: 6 | fallbacks: 0
  [   7/1000] 0.0s | retrieval sim=0.49 | 0.0 min elapsed | ETA 0.5 min | retrieved: 7 | fallbacks: 0
  [   8/1000] 0.0s | retrieval sim=0.46 | 0.0 min elapsed | ETA 0.5 min | retrieved: 8 | fallbacks: 0
  [   9/1000] 0.0s | retrieval sim=0.29 | 0.0 min elapsed | ETA 0.5 min | retrieved: 9 | fallbacks: 0
  [  10/1000] 0.0s | retrieval sim=0.45 | 0.0 min elapsed | ETA 0.

,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" clip-r..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
